# Experiment — Imputation: median vs mean vs KNN

**Category:** imputation | **Tasks 2 & 3**

**Hypothesis:** the imputation method changes downstream model accuracy. We compare mean, median and KNN imputation by their effect on a fixed RandomForest's test RMSE — i.e. we judge imputation by *model impact*, not by how complete the table looks.

In [1]:
import sys, warnings
from pathlib import Path
warnings.filterwarnings("ignore")
REPO = Path.cwd()
while not (REPO / "src").exists() and REPO != REPO.parent:
    REPO = REPO.parent
sys.path.insert(0, str(REPO))
import numpy as np
import pandas as pd
pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 40)
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from src.data_loading import load_dataset, TARGET
from src.preprocessing import coerce_types, enforce_ranges, impute
from src.features import build_features, split_xy
from src.evaluation import regression_metrics

In [2]:
raw = load_dataset()
# coerce types + null out-of-range values so all gaps go through imputation
base = enforce_ranges(coerce_types(raw).drop(columns=['Record_ID']))
print('Missing before imputation:')
base.isna().sum()[base.isna().sum() > 0]

Missing before imputation:


Hour                    1
Temperature_C           2
Humidity_pct            2
Rainfall_mm             4
PopulationIndex         1
IndustrialIndex         1
SolarGenerationIndex    3
GridLoad_MW             1
dtype: int64

## Compare strategies on downstream RMSE
Same split, same model, only the imputation differs.

In [3]:
def score_strategy(strategy):
    d = impute(base, strategy=strategy)
    feat = build_features(d)
    X, y = split_xy(feat)
    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, random_state=42)
    m = RandomForestRegressor(n_estimators=300, random_state=42, n_jobs=-1)
    m.fit(Xtr, ytr)
    return regression_metrics(yte, m.predict(Xte))

results = {s: score_strategy(s) for s in ['mean', 'median', 'knn']}
board = pd.DataFrame(results).T.sort_values('RMSE').round(3)
board

,RMSE,MAE,R2,MAPE
knn,18.931,14.807,0.965,2.484
median,18.944,14.810,0.965,2.482
mean,18.952,14.830,0.965,2.485


In [4]:
best = board.index[0]
print(f'Best imputation by RMSE: {best}')

Best imputation by RMSE: knn


## Decision (Task 2 justification)
We pick the strategy with the lowest downstream RMSE above. **Median** is the default used across the project because it is robust to the outliers/junk seen in Task 1 and performs on par with KNN while being far cheaper and fully reproducible. Record the winning metric in `experiment_log.md`.